In [1]:
!git clone https://github.com/nlihin/R2R-analysis

fatal: destination path 'R2R-analysis' already exists and is not an empty directory.


In [2]:
import os
import pandas as pd
import numpy as np
from scipy.stats import linregress

In [3]:
%load_ext autoreload 
%autoreload 2  
import iol_utils as iu   

In [4]:
BASE_PATH = "R2R-analysis/data"
CASE_STUDIES = ["case_study_1", "case_study_2", "case_study_3"]
DATA_TYPES = ["ratings", "IOL", "more_qs"]

## Load data

In [5]:
ratings = iu.load_case_study_files(BASE_PATH, CASE_STUDIES, "ratings", min_session=7)
iol = iu.load_case_study_files(BASE_PATH, CASE_STUDIES, "IOL")
more_qs = iu.load_case_study_files(BASE_PATH, CASE_STUDIES, "more_qs")

Processing folder: R2R-analysis/data\case_study_1\ratings
Reading first file with headers: R2R-analysis/data\case_study_1\ratings\10.csv
Skipping file: 1_rate.csv
Skipping file: 2_rate.csv
Skipping file: 3_rate.csv
Skipping file: 4_rate.csv
Skipping file: 5_rate.csv
Skipping file: 6_rate.csv
Reading file without headers: R2R-analysis/data\case_study_1\ratings\7_rate.csv
Reading file without headers: R2R-analysis/data\case_study_1\ratings\8_rate.csv
Reading file without headers: R2R-analysis/data\case_study_1\ratings\9_rate.csv
Processing folder: R2R-analysis/data\case_study_2\ratings
Reading file without headers: R2R-analysis/data\case_study_2\ratings\11_rate.csv
Reading file without headers: R2R-analysis/data\case_study_2\ratings\12_rate.csv
Reading file without headers: R2R-analysis/data\case_study_2\ratings\13_rate.csv
Reading file without headers: R2R-analysis/data\case_study_2\ratings\14_rate.csv
Reading file without headers: R2R-analysis/data\case_study_2\ratings\15_rate.csv
Read

In [6]:
unique_usernames_count = ratings["username"].nunique()
print("Number of unique usernames in ratings from session 7 and above:", unique_usernames_count)

Number of unique usernames in ratings from session 7 and above: 270


In [7]:
more_qs.head()

,username,question_number,answer,group_number,session_number
0,uid12_10,1,3,3,10
1,uid12_10,2,4,3,10
2,uid12_10,3,3,3,10
3,uid14_10,1,4,3,10
4,uid14_10,2,5,3,10


## Rating-pattern feature calculation



In [8]:
ratings_stats = iu.calculate_pattern_stats(
    df=ratings,
    value_col="rate",
    time_col="time",
    prefix=""
)

C:\Users\lihin\OneDrive - ariel.ac.il\Ariel_Research\R2R\my code\naama_IOL1\iol_utils.py:72: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["timestamp"] = pd.to_datetime(data[time_col], errors="coerce")


In [9]:
understand_project_stats = iu.calculate_pattern_stats(
    df=more_qs[more_qs['question_number']==3],
    value_col="answer",
    time_col=None,
    prefix="understand_project_"
)

## Reverse columns in questionnaire

In [10]:
cols_1to5 = ["satf4", "satf5", "satf6"]
cols_1to7 = ["fatig3", "fatig5"]
missing_1to5 = [c for c in cols_1to5 if c not in iol.columns]
missing_1to7 = [c for c in cols_1to7 if c not in iol.columns]
if missing_1to5 or missing_1to7:
    print("columns not found", {"1-5": missing_1to5, "1-7": missing_1to7})

for col in cols_1to5:
    if col in iol.columns:
        orig = iol[col].copy()
        v = pd.to_numeric(iol[col], errors="coerce")
        rev = 6 - v
        mask = v.between(1, 5)
        iol[col] = np.where(v.notna() & mask, rev, orig)

for col in cols_1to7:
    if col in iol.columns:
        orig = iol[col].copy()
        v = pd.to_numeric(iol[col], errors="coerce")
        rev = 8 - v
        mask = v.between(1, 7)
        iol[col] = np.where(v.notna() & mask, rev, orig)

## Merge the rating-pattern features with IOL


In [11]:
all_data = (
    iol
    .merge(ratings_stats, on="username", how="inner")
    .merge(understand_project_stats, on="username", how="inner")
)

In [12]:
all_data = all_data.dropna(subset=["patrn1"])

## Save the final dataset

In [13]:
all_data.to_csv("all_data_to_sem.csv", index=False)